In [2]:
import pandas as pd
import xarray as xr
from pathlib import Path

path = "/Users/vb/Downloads/20250531_Delft-davis-gv.nc"
ds = xr.open_dataset(path)
print("Variables in dataset:", list(ds.data_vars))
print("Columns in dataframe:", df.columns.tolist()[:50])

Variables in dataset: ['latitude', 'longitude', 'altitude', 'barometer', 'inHumidity', 'inTemp', 'outHumidity', 'outTemp', 'radiation', 'rain', 'rainRate', 'UV', 'windDir', 'windGust', 'windGustDir', 'windSpeed']
Columns in dataframe: ['time', 'latitude', 'longitude', 'altitude', 'barometer', 'inHumidity', 'inTemp', 'outHumidity', 'outTemp', 'radiation', 'rain', 'rainRate', 'UV', 'windDir', 'windGust', 'windGustDir', 'windSpeed']


In [3]:
import pandas as pd
import xarray as xr
from pathlib import Path

path = "/Users/vb/Downloads/20250531_Delft-davis-gv.nc"
ds = xr.open_dataset(path)

# Convert to DataFrame
df = ds.to_dataframe().reset_index()

# --- Map dataset variable names -> your canonical names ---
name_map = {
    "barometer": "pressure",
    "outTemp": "external_temp",          # external temperature
    "outHumidity": "humidity",
    "radiation": "solar_irradiance",
    "windSpeed": "wind_speed",
    # (available but not used now): "inTemp", "inHumidity", "windDir", "windGust", "windGustDir", "UV", "rain", "rainRate"
}

# Keep only columns that actually exist
src_cols = [c for c in name_map.keys() if c in df.columns]
if not src_cols:
    raise ValueError("None of the expected sensor columns were found in the dataset.")

# Build a working frame with time + selected sensors
ext_raw = df[["time"] + src_cols].copy()

# --- Build timestamp_utc from 'time' ---
# xarray's 'time' is typically np.datetime64[ns] and already UTC-like; coerce to pandas datetime with UTC
ext_raw["timestamp_utc"] = pd.to_datetime(ext_raw["time"], utc=True, errors="coerce")

# --- Rename to canonical names and ensure numeric dtypes ---
ext_raw = ext_raw.rename(columns=name_map)
sensor_cols = [name_map[c] for c in src_cols]  # canonical names that are present

for c in sensor_cols:
    ext_raw[c] = pd.to_numeric(ext_raw[c], errors="coerce")

# --- Basic cleanup: drop bad timestamps, dedupe on time+values, sort ---
subset_for_dupes = ["timestamp_utc"] + sensor_cols
ext_clean = (
    ext_raw.dropna(subset=["timestamp_utc"])
           .drop_duplicates(subset=subset_for_dupes)
           .sort_values("timestamp_utc")
           .reset_index(drop=True)
)

# --- Hourly resample (mean) ---
ext_hourly = (
    ext_clean.set_index("timestamp_utc")[sensor_cols]
             .resample("1h")
             .mean()              # NaNs ignored; an all-NaN hour stays NaN
             .reset_index()
)

# Local time (Europe/Amsterdam), if you want it
ext_hourly["time_local"] = ext_hourly["timestamp_utc"].dt.tz_convert("Europe/Amsterdam")

print("External hourly head():")
print(ext_hourly.head())
print(f"Total rows in ext_hourly: {len(ext_hourly)}")
print("Counts per column (non-NaN):")
print(ext_hourly[sensor_cols].count())

External hourly head():
              timestamp_utc     pressure  external_temp   humidity  \
0 2025-05-31 00:00:00+00:00  1018.622131      13.577778  86.000000   
1 2025-05-31 01:00:00+00:00  1018.600708      13.282408  86.183334   
2 2025-05-31 02:00:00+00:00  1018.343384      13.380555  87.000000   
3 2025-05-31 03:00:00+00:00  1018.126648      13.482408  87.000000   
4 2025-05-31 04:00:00+00:00  1017.971985      13.647222  87.000000   

   solar_irradiance  wind_speed                time_local  
0          0.000000         0.0 2025-05-31 02:00:00+02:00  
1          0.000000         0.0 2025-05-31 03:00:00+02:00  
2          0.000000         0.0 2025-05-31 04:00:00+02:00  
3          2.266667         0.0 2025-05-31 05:00:00+02:00  
4         25.983334         0.0 2025-05-31 06:00:00+02:00  
Total rows in ext_hourly: 24
Counts per column (non-NaN):
pressure            24
external_temp       24
humidity            24
solar_irradiance    24
wind_speed          24
dtype: int64


In [5]:
# saving file into an excel
# Making a copy and dropping timezone info from all datetime-with-tz columns
df_xlsx = ext_hourly.copy()
for c in df_xlsx.select_dtypes(include=["datetimetz"]).columns:
    # keep in UTC, then remove tz info
    df_xlsx[c] = df_xlsx[c].dt.tz_convert("UTC").dt.tz_localize(None)
# Now write to Excel
df_xlsx.to_excel("TUDsensorCheck.xlsx", index=False)